# SymbioPan — Full Project Tour

This notebook imports and demonstrates **every module** in the SymbioPan codebase for the PUMA Grand Challenge Track 2 panoptic segmentation pipeline.

## Package Structure
- `configs/` — Centralized configuration dataclasses
- `data/` — Constants, preprocessing, dataset, transforms, sampling
- `models/` — Encoder, backbone, FPN, decoders, cross-attention, panoptic net, Stage 2 refiner
- `training/` — Train loop, checkpoint, logging, CLI, Stage 1 & 2 trainers
- `inference/` — WSI tiling, model loading, site classifier, cellpose flow, postprocessing
- `utils/` — Losses, metrics, spatial prior, SC-DFA, split utils, normalization
- `scripts/` — Entry-point scripts

In [ ]:
import sys, torch, numpy as np
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

---
## 1. configs/ — Configuration

In [ ]:
from configs import (
    PATHS, INFERENCE_DEFAULT_CONFIG, PREPROCESS_DEFAULT_CONFIG,
    STAGE1_DEFAULT_CONFIG, STAGE2_DEFAULT_CONFIG,
    InferenceConfig, PathsConfig, PreprocessConfig, Stage1Config, Stage2Config,
)
from configs.defaults import get_device, linear_ramp
from configs.serialization import make_inference_config_from_stage1

print("Paths:", PATHS)
print("Stage 1 batch_size:", STAGE1_DEFAULT_CONFIG.batch_size)
print("Preprocess image_size:", PREPROCESS_DEFAULT_CONFIG.image_size)
print("get_device():", get_device())
print("linear_ramp(epoch=12, start=10, end=16, max=0.5):", linear_ramp(12, 10, 16, 0.5))

---
## 2. data/ — Constants, Preprocessing, Dataset

### 2a. data.constants — Label mappings & weights

In [ ]:
from data.constants import (
    PUMA_TISSUE_ID_TO_NAME, INTERNAL_TISSUE_ID_TO_NAME, NUM_TISSUE_CLASSES,
    PUMA_NUCLEI_ID_TO_NAME, NUM_NUCLEI_CLASSES,
    RARE_TISSUE_IDS, RARE_NUCLEI_IDS,
    TISSUE_CLASS_WEIGHTS, NUCLEI_CLASS_WEIGHTS, STAGE2_NUCLEI_WEIGHTS,
    LOSS_MULTIPLIERS, HV_GRAD_THRESHOLD,
    NORMALIZATION_MEAN, NORMALIZATION_STD, IGNORE_INDEX,
)

print(f"Tissue classes ({NUM_TISSUE_CLASSES}):", INTERNAL_TISSUE_ID_TO_NAME)
print(f"Nuclei classes ({NUM_NUCLEI_CLASSES}):", PUMA_NUCLEI_ID_TO_NAME)
print(f"Tissue weights:", TISSUE_CLASS_WEIGHTS)
print(f"Nuclei weights:", NUCLEI_CLASS_WEIGHTS)
print(f"Normalization:", NORMALIZATION_MEAN, NORMALIZATION_STD)

### 2b. data.preprocessing — GeoJSON parsing, flow generation

In [ ]:
from data.preprocessing.geojson_parser import parse_geojson_masks, find_annotation_file
from data.preprocessing.flow_generator import CellposeFlowGenerator, compute_hv_map, compute_hv_map_torch
from data.preprocessing import main as preprocess_main

# Demonstrate helper functions with dummy data
dummy_inst = np.zeros((64, 64), dtype=np.int32)
dummy_inst[10:20, 10:30] = 1
dummy_inst[40:50, 40:55] = 2
hv_map = compute_hv_map(dummy_inst)
print(f"compute_hv_map output shape: {hv_map.shape}")

dummy_tensor = torch.from_numpy(dummy_inst)
hv_torch = compute_hv_map_torch(dummy_tensor)
print(f"compute_hv_map_torch output shape: {hv_torch.shape}")

# CellposeFlowGenerator (disabled to avoid loading cellpose)
cp_gen = CellposeFlowGenerator(enabled=False, model_type="nuclei")
dummy_image = np.zeros((64, 64, 3), dtype=np.uint8)
flow = cp_gen.make_flow(dummy_image)
print(f"CellposeFlowGenerator (disabled) output shape: {flow.shape}")

### 2c. data.dataset — Dataset, transforms, sampling

In [ ]:
from data.dataset import PUMADataset, get_train_transforms, get_val_transforms
from data.dataset.puma_dataset import puma_tissue_to_internal, internal_tissue_to_puma, source_name_from_base_name
from data.dataset.sampling import compute_sample_weight, compute_all_sample_weights
from data.dataset.transforms import VectorSafeCompose

# Demonstrate label conversion
puma_labels = np.array([0, 1, 2, 3, 4, 5], dtype=np.uint8)
internal = puma_tissue_to_internal(puma_labels)
back_to_puma = internal_tissue_to_puma(internal)
print(f"PUMA -> internal: {puma_labels} -> {internal}")
print(f"internal -> PUMA: {internal} -> {back_to_puma}")

# Source name extraction
print(f"source_name_from_base_name('roi01__rare00_tissue2'): {source_name_from_base_name('roi01__rare00_tissue2')}")

# Transforms
train_tf = get_train_transforms(1024)
val_tf = get_val_transforms(1024)
print(f"Train transforms: {type(train_tf).__name__}")
print(f"Val transforms: {type(val_tf).__name__}")

### 2d. Try loading the processed dataset (if available)

In [ ]:
data_dir = PATHS.data_dir
try:
    ds = PUMADataset(data_dir, transforms=get_val_transforms(1024))
    print(f"Dataset loaded: {len(ds)} samples")
    batch = ds[0]
    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            print(f"  {k}: {v.shape}")
        else:
            print(f"  {k}: {v}")
except Exception as e:
    print(f"Dataset not available: {e}")

---
## 3. models/ — Neural Network Architectures

### 3a. backbone — ConvNeXt-Atto

In [ ]:
from models import build_cnn_backbone, UnifiedPanopticNet, ResidualNucleiRefinerUNet, build_stage2_input
from models.backbone import get_cnn_spatial_prior
from models.encoder import build_uni_vit, get_frozen_uni_model, UnifiedPanopticEncoder
from models.cross_attention import SpatialInjector
from models.fpn_aggregator import FPNAggregator
from models.decoders import (
    ParallelDecoders, MutualFeatureExchange,
    HoVerNeXtNucleiHead, ASPP, ASPPBranch,
)
from models.panoptic_net import UnifiedPanopticNet
from models.stage2_refiner import ResidualNucleiRefinerUNet, build_stage2_input

# Build CNN backbone
cnn = build_cnn_backbone(pretrained=False)
print(f"CNN backbone: {type(cnn).__name__}")
if hasattr(cnn, 'feature_info'):
    print(f"  feature channels: {cnn.feature_info.channels()}")

# FPN
fpn = FPNAggregator(vit_dim=1024, cnn_dims=[40, 80, 160, 320], fpn_dim=256)
x = torch.randn(1, 257, 1024)
cnn_feats = [torch.randn(1, d, 256//(2**i), 256//(2**i)) for i, d in enumerate([40, 80, 160, 320])]
fpn_out = fpn(x, cnn_feats)
print(f"FPN output keys: {list(fpn_out.keys())}")
for k, v in fpn_out.items():
    print(f"  {k}: {v.shape}")

# SpatialInjector
injector = SpatialInjector(vit_dim=1024, cnn_dims=[40, 80, 160, 320])
vit_tokens = torch.randn(1, 257, 1024)
cnn_feats = [torch.randn(1, d, 64, 64) for d in [40, 80, 160, 320]]
out = injector(vit_tokens, cnn_feats)
print(f"SpatialInjector output shape: {out.shape}")

# MutualFeatureExchange
mfe = MutualFeatureExchange(dim=256)
ft, fn = mfe(torch.randn(1, 256, 32, 32), torch.randn(1, 256, 32, 32))
print(f"MFE output shapes: ft={ft.shape}, fn={fn.shape}")

# ParallelDecoders
dec = ParallelDecoders(fpn_dim=256, num_tissue=5, num_nuclei=10)
fpn_feats = {k: torch.randn(1, 256, s, s) for k, s in zip(["p1","p2","p3","p4","p5"], [128, 64, 32, 16, 8])}
t, np_, nc, hv = dec(fpn_feats, torch.randn(1, 2, 128, 128))
print(f"Decoder outputs: tissue={t.shape}, np={np_.shape}, nc={nc.shape}, hv={hv.shape}")

# ASPP & HoVerNeXt head
aspp = ASPP(256, 256)
print(f"ASPP: {aspp(torch.randn(1, 256, 32, 32)).shape}")
hn = HoVerNeXtNucleiHead(258, 64, 2)
print(f"HoVerNeXt head: {hn(torch.randn(1, 258, 128, 128)).shape}")

### 3b. Stage 2 Refiner

In [ ]:
# ResidualNucleiRefinerUNet
s2 = ResidualNucleiRefinerUNet(in_channels=21, out_classes=10)
s2_in = torch.randn(1, 21, 128, 128)
s2_out = s2(s2_in)
print(f"Stage 2 refiner output shape: {s2_out.shape}")

# Verify zero-init
print(f"Final conv zero-init: weight sum = {s2.outc.weight.abs().sum().item():.6f}")

### 3c. Full UnifiedPanopticNet (Stage 1)

In [ ]:
# Build the full model (no UNI weights for demo)
cnn = build_cnn_backbone(pretrained=False)
model = UnifiedPanopticNet(
    vit_model=None,
    cnn_model=cnn,
    num_tissue=5,
    num_nuclei=10,
    load_uni_weights=False,
)

# Forward pass with dummy data
model.eval()
images = torch.randn(1, 3, 256, 256)
cp_flows = torch.randn(1, 2, 256, 256)
with torch.no_grad():
    out = model(images, cp_flows, site_types=None)
print("UnifiedPanopticNet outputs:")
for k, v in out.items():
    print(f"  {k}: {v.shape}")

# SC-DFA and spatial prior
print(f"\nSC-DFA available: {hasattr(model, 'sc_dfa')}")
print(f"Spatial prior available: {hasattr(model, 'spatial_prior')}")
model.enable_sc_dfa(True)
model.set_sc_dfa_lambda(0.3)
model.set_spatial_prior_lambda(0.2)
print(f"SC-DFA lambda: {model.lambda_sc_dfa}, Prior lambda: {model.lambda_prior}")

---
## 4. training/ — Training Pipeline

In [ ]:
from training import (
    stage1_main, stage2_main,
    train_one_epoch, validate,
    safe_torch_save, load_large_checkpoint, extract_state_dict,
    logger, setup_logger,
)
from training.cli import parse_stage1_args, parse_stage2_args
from training.train_loop import _batch_to_device, _autocast_context
from training.checkpoint import safe_torch_save, load_large_checkpoint, extract_state_dict

print(f"Logger: {logger.name} (level={logger.level})")
print(f"safe_torch_save available: {callable(safe_torch_save)}")
print(f"extract_state_dict available: {callable(extract_state_dict)}")
print(f"train_one_epoch / validate: {callable(train_one_epoch)} / {callable(validate)}")

---
## 5. utils/ — Losses, Metrics, Priors, Split

### 5a. Losses

In [ ]:
from utils.losses import (
    MultiTaskUncertaintyLoss, SafeCrossEntropyLoss,
    FocalTverskyLoss, SoftDiceLoss, FocalBCELoss,
)

batch, H, W = 2, 32, 32
preds = {
    "tissue": torch.randn(batch, 5, H, W),
    "np": torch.randn(batch, 1, H, W),
    "nc": torch.randn(batch, 10, H, W),
    "hv": torch.randn(batch, 2, H, W),
}
targets = {
    "tissue_sem": torch.randint(0, 5, (batch, H, W), dtype=torch.long),
    "nuclei_nc": torch.randint(0, 10, (batch, H, W), dtype=torch.long),
    "nuclei_np": torch.randint(0, 2, (batch, H, W), dtype=torch.long),
    "nuclei_hv": torch.randn(batch, 2, H, W),
}

criterion = MultiTaskUncertaintyLoss()
total, branch_losses = criterion(preds, targets)
print(f"MultiTaskUncertaintyLoss: total={total.item():.4f}, branches={branch_losses}")

# Individual losses
ce = SafeCrossEntropyLoss()(torch.randn(2, 5, H, W), targets["tissue_sem"])
ft = FocalTverskyLoss()(torch.randn(2, 5, H, W), targets["tissue_sem"])
print(f"SafeCE: {ce.item():.4f}, FocalTversky: {ft.item():.4f}")

### 5b. Metrics

In [ ]:
from utils.metrics import PUMAMetrics, SemanticMetricAccumulator

metrics = PUMAMetrics()
result = metrics.calculate_all_metrics(preds, targets)
print("PUMAMetrics output:")
for k, v in result.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

### 5c. Spatial Prior & SC-DFA

In [ ]:
from utils.priors import SpatialLogitAdjuster
from utils.sc_dfa import SCDFA

# SC-DFA
sc_dfa = SCDFA(num_tissue_classes=5, num_nuclei_classes=10)
tissue_logits = torch.randn(1, 5, 32, 32)
nc_bias = sc_dfa(tissue_logits)
print(f"SC-DFA bias shape: {nc_bias.shape}")

# SpatialLogitAdjuster
prior = SpatialLogitAdjuster(num_tissue_classes=5, num_nuclei_classes=10)
nc_logits = torch.randn(1, 10, 32, 32)
adjusted = prior(nc_logits, tissue_logits, "metastatic", lambda_scale=0.2)
print(f"SpatialPrior adjusted shape: {adjusted.shape}")

### 5d. Split Utils & Normalization

In [ ]:
from utils.split_utils import make_or_load_group_split
from utils.normalization import normalize_image

# Normalization demo
dummy_img = np.random.randint(0, 256, (64, 64, 3), dtype=np.uint8)
normed = normalize_image(dummy_img)
print(f"Normalized image shape: {normed.shape}, dtype: {normed.dtype}")
print(f"  mean: {normed.mean():.4f}, std: {normed.std():.4f}")

---
## 6. inference/ — WSI Inference Pipeline

In [ ]:
from inference import main as inference_main
from inference.tiling import (
    find_single_tif, read_rgb_uint8, normalize_tile,
    make_tile_starts, pad_reflect, autocast_enabled,
)
from inference.model_loader import load_stage1, load_stage2
from inference.site_classifier import load_site_classifier, predict_site_type, resolve_site_type
from inference.cellpose_flow import CellposeFlowGenerator as InferenceCellposeFlowGenerator
from inference.postprocessing import (
    hv_instance_segmentation, classify_instances, instances_to_polygons,
)
from inference.infer_wsi import DefaultSTAGE1_CP as INFER_STAGE1_CP

print(f"Inference main: {inference_main}")
print(f"make_tile_starts for len=2000, tile=1024, stride=768:"
      f" {make_tile_starts(2000, 1024, 768)}")

# pad_reflect demo
small_tile = np.random.randint(0, 256, (512, 768, 3), dtype=np.uint8)
padded, rh, rw = pad_reflect(small_tile, 1024)
print(f"pad_reflect: {small_tile.shape} -> {padded.shape} (real_h={rh}, real_w={rw})")

---
## 7. scripts/ — Entry Points

In [ ]:
from scripts.run_preprocess import main as script_preprocess
from scripts.run_stage1 import main as script_stage1
from scripts.run_stage2 import main as script_stage2
from scripts.run_inference import main as script_inference

print("All script entry points imported:")
print(f"  run_preprocess: {script_preprocess}")
print(f"  run_stage1:     {script_stage1}")
print(f"  run_stage2:     {script_stage2}")
print(f"  run_inference:  {script_inference}")

---
## 8. Verify All Exports from __init__.py Files

In [ ]:
print("=== data.__init__ ===")
from data import (
    HV_GRAD_THRESHOLD, INTERNAL_TISSUE_ID_TO_NAME, LOSS_MULTIPLIERS,
    NORMALIZATION_MEAN, NORMALIZATION_STD, NUCLEI_CLASS_WEIGHTS,
    NUM_NUCLEI_CLASSES, NUM_TISSUE_CLASSES,
    PUMA_NUCLEI_ID_TO_NAME, PUMA_NUCLEI_NAME_TO_ID,
    PUMA_TISSUE_ID_TO_NAME, PUMA_TISSUE_NAME_TO_ID,
    RARE_NUCLEI_IDS, RARE_NUCLEI_SAMPLE_BONUS,
    RARE_TISSUE_IDS, RARE_TISSUE_IDS_PUMA, RARE_TISSUE_SAMPLE_BONUS,
    STAGE2_NUCLEI_WEIGHTS, TISSUE_CLASS_WEIGHTS,
)
print("  All data constants imported OK")

print("\n=== models.__init__ ===")
from models import build_cnn_backbone, UnifiedPanopticNet, ResidualNucleiRefinerUNet, build_stage2_input
print("  All models imported OK")

print("\n=== utils.__init__ ===")
from utils import MultiTaskUncertaintyLoss, PUMAMetrics, SpatialLogitAdjuster, SCDFA
print("  All utils imported OK")

print("\n=== training.__init__ ===")
from training import extract_state_dict, load_large_checkpoint, safe_torch_save, logger, setup_logger, stage1_main, stage2_main, train_one_epoch, validate
print("  All training imported OK")

print("\n=== All modules imported successfully ===")